In [1]:
import pandas as pd
import numpy as np
import holidays
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error
import os
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display
import matplotlib.pyplot as plt
import holidays

In [2]:
RAW_CSV = r"C:\Users\fdcontreras\OneDrive - Indra\Universidad\Despliegue de Soluciones Analíticas\Proyecto_Despliegue\Proyecto_Despliegue\data\02_processed\olist_consolidated_dataset.csv"

df = pd.read_csv(RAW_CSV, parse_dates=["order_purchase_timestamp"])
# columna order_date
df["order_date"] = df["order_purchase_timestamp"].dt.date

daily = (
    df.groupby(["product_category_name_english","order_date"])  
      .size().reset_index(name="units_sold")
)

In [4]:
category_days = (
    df
    .groupby("product_category_name_english")["order_date"]
    .nunique()                             # cuenta de fechas únicas
    .reset_index(name="unique_days")       # renombra la columna
    .sort_values("unique_days", ascending=False)
)

# (Opcional) Para ver todo el resultado sin cortes:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)
display(category_days)

,product_category_name_english,unique_days
39,furniture_decor,607
65,sports_leisure,603
43,health_beauty,602
7,bed_bath_table,598
15,computers_accessories,593
69,toys,590
68,telephony,588
49,housewares,587
42,garden_tools,585
20,cool_stuff,584


In [6]:
MIN_UNIQUE_DAYS = 45

valid_cats_df = category_days[
    category_days["unique_days"] >= MIN_UNIQUE_DAYS
].copy()

display(valid_cats_df)

valid_cats = valid_cats_df["product_category_name_english"].tolist()
print(f"Se seleccionaron {len(valid_cats)} categorías con ≥{MIN_UNIQUE_DAYS} días de ventas.")


,product_category_name_english,unique_days
39,furniture_decor,607
65,sports_leisure,603
43,health_beauty,602
7,bed_bath_table,598
15,computers_accessories,593
69,toys,590
68,telephony,588
49,housewares,587
42,garden_tools,585
20,cool_stuff,584


Se seleccionaron 58 categorías con ≥45 días de ventas.


In [7]:
daily_orig = (
    df
    .query("product_category_name_english in @valid_cats") 
    .groupby(["product_category_name_english", "order_date"])
    .size()
    .reset_index(name="units_sold")
)

all_dates = pd.date_range(
    start = daily_orig["order_date"].min(),
    end   = daily_orig["order_date"].max(),
    freq  = "D"
)
all_cats = daily_orig["product_category_name_english"].unique()
full_index = pd.MultiIndex.from_product(
    [all_cats, all_dates],
    names=["product_category_name_english", "order_date"]
)

daily = (
    daily_orig
    .set_index(["product_category_name_english", "order_date"])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

daily.head()

,product_category_name_english,order_date,units_sold
0,agro_industry_and_commerce,2016-09-04,0
1,agro_industry_and_commerce,2016-09-05,0
2,agro_industry_and_commerce,2016-09-06,0
3,agro_industry_and_commerce,2016-09-07,0
4,agro_industry_and_commerce,2016-09-08,0


In [8]:
TRAIN_DAYS     = 30      
TEST_DAYS      = 15      
TS_SPLITS      = 5
OOB_CANDIDATES = [100, 200, 500]
RANDOM_STATE   = 42


years = sorted(daily['order_date'].apply(lambda d: d.year).unique())
br_hols = holidays.Brazil(years=years)

results = []
tscv = TimeSeriesSplit(n_splits=TS_SPLITS)

In [9]:
cat = "health_beauty"
df_cat = daily.query("product_category_name_english == @cat").copy()
df_cat['order_date'] = pd.to_datetime(df_cat['order_date'])
df_cat.set_index('order_date', inplace=True)

# 1) Feature engineering
df_cat['lag1']    = df_cat['units_sold'].shift(1)
df_cat['roll7']   = df_cat['units_sold'].rolling(7).mean()
idx = df_cat.index
df_cat['dow_sin'] = np.sin(2*np.pi * idx.weekday / 7)
df_cat['dow_cos'] = np.cos(2*np.pi * idx.weekday / 7)
df_cat['mes_sin'] = np.sin(2*np.pi * idx.month   / 12)
df_cat['mes_cos'] = np.cos(2*np.pi * idx.month   / 12)
df_cat['festivo']= idx.to_series().isin(br_hols).astype(int)

# Black Friday
bf_dates = []
for y in idx.year.unique():
    nov = pd.date_range(f"{y}-11-01", f"{y}-11-30", freq="D")
    th  = nov[nov.weekday == 3]
    bf_dates.append((th[3] + pd.Timedelta(days=1)).date())
df_cat['bf'] = df_cat.index.to_series().apply(lambda d: d.date() in bf_dates).astype(int)

df_feat = df_cat.dropna().drop(columns=['product_category_name_english'])
if len(df_feat) > TRAIN_DAYS and df_feat['units_sold'].iloc[TRAIN_DAYS:TRAIN_DAYS+TEST_DAYS].sum()>0:
    # 2) Split
    X = df_feat.drop(columns=['units_sold'])
    y = df_feat['units_sold'].values
    X_train, X_test = X.iloc[:TRAIN_DAYS], X.iloc[TRAIN_DAYS:TRAIN_DAYS+TEST_DAYS]
    y_train, y_test = y[:TRAIN_DAYS], y[TRAIN_DAYS:TRAIN_DAYS+TEST_DAYS]

    # 3) OOB para elegir n_estimators
    best_oob, best_n = -np.inf, None
    for n in OOB_CANDIDATES:
        tmp = RandomForestRegressor(n_estimators=n, oob_score=True,
                                    random_state=RANDOM_STATE, n_jobs=-1)
        tmp.fit(X_train, y_train)
        if tmp.oob_score_ > best_oob:
            best_oob, best_n = tmp.oob_score_, n

    # 4) RandomizedSearchCV
    param_dist = {
        'n_estimators':    [max(5, best_n//2), best_n, best_n*2],
        'max_depth':       [None, 5, 10],
        'min_samples_leaf':[1, 2, 4],
        'max_features':    ['sqrt', 0.5]
    }
    search = RandomizedSearchCV(
        RandomForestRegressor(random_state=RANDOM_STATE, oob_score=True),
        param_distributions=param_dist,
        n_iter=20,
        cv=TimeSeriesSplit(n_splits=TS_SPLITS),
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        random_state=RANDOM_STATE
    )
    search.fit(X_train, y_train)
    best_model = search.best_estimator_

    # 5) Métricas
    preds = best_model.predict(X_test)
    mae   = mean_absolute_error(y_test, preds)
    rmse  = np.sqrt(mean_squared_error(y_test, preds))
    mape  = np.mean(np.abs((y_test - preds) / y_test)[y_test > 0]) * 100

    # 6) Mostrar resultado en tabla
    res = pd.DataFrame([{
        'category':       cat,
        'best_n_oob':     best_n,
        'init_oob_score': best_oob,
        'MAE':            mae,
        'RMSE':           rmse,
        'MAPE (%)':       mape,
        **search.best_params_
    }])
    pd.set_option('display.max_columns', None)
    display(res)

,category,best_n_oob,init_oob_score,MAE,RMSE,MAPE (%),n_estimators,min_samples_leaf,max_features,max_depth
0,health_beauty,500,0.613835,2.932533,3.999043,3.666667,250,1,0.5,10


In [ ]:
health_beauty_train = X_train.copy()
health_beauty_train["units_sold"] = y_train

health_beauty_test  = X_test.copy()
health_beauty_test ["units_sold"] = y_test

health_beauty_train = health_beauty_train.reset_index()
health_beauty_test  = health_beauty_test.reset_index()

from IPython.display import display
display(health_beauty_train.head())
display(health_beauty_test.head())

health_beauty_train.to_csv("health_beauty_train.csv", index=False)
health_beauty_test.to_csv("health_beauty_test.csv",  index=False)

,order_date,lag1,roll7,dow_sin,dow_cos,mes_sin,mes_cos,festivo,bf,units_sold
0,2016-09-10,0.0,0.0,-0.974928,-0.222521,-1.0,-1.836970e-16,0,0,0
1,2016-09-11,0.0,0.0,-0.781831,0.623490,-1.0,-1.836970e-16,0,0,0
2,2016-09-12,0.0,0.0,0.000000,1.000000,-1.0,-1.836970e-16,0,0,0
3,2016-09-13,0.0,0.0,0.781831,0.623490,-1.0,-1.836970e-16,0,0,0
4,2016-09-14,0.0,0.0,0.974928,-0.222521,-1.0,-1.836970e-16,0,0,0


,order_date,lag1,roll7,dow_sin,dow_cos,mes_sin,mes_cos,festivo,bf,units_sold
0,2016-10-10,6.0,7.142857,0.000000,1.000000,-0.866025,0.5,0,0,6
1,2016-10-11,6.0,5.571429,0.781831,0.623490,-0.866025,0.5,0,0,0
2,2016-10-12,0.0,4.714286,0.974928,-0.222521,-0.866025,0.5,1,0,0
3,2016-10-13,0.0,3.571429,0.433884,-0.900969,-0.866025,0.5,0,0,0
4,2016-10-14,0.0,2.571429,-0.433884,-0.900969,-0.866025,0.5,0,0,0
